# Issue #232 — causal SP-backed divergence support (BJ-UNIFIED-DIVERGENCE-SUPPORT-001)

Candidate-mode data remediation. This notebook is a thin readable
orchestration layer: it runs the reusable `src/divergence_support` package
top-to-bottom, renders every committed machine-readable table and verifies
the immutable package. All durable evidence lives in
`results/bj232_divergence_support_v1/`.

The controlling question: do existing or recoverable pre-event sources
produce enough issue-level causal SP support for #209 chronological
unified-model development?

## Setup

Run from the repository root with the project's pinned environment
(`PYTHONPATH=src .venv/bin/python`). The package reads only the accepted
upstream frozen packages (#203/#204/#214/#180), each SHA256-verified before
use; no settlement/result table is opened. Raw gitignored assets are
census-recorded when present and reported ABSENT when not.

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "results").exists():
    ROOT = ROOT.parents[2]
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from divergence_support import builder, contract

PACKAGE = contract.package_path(ROOT)
print("root:", ROOT)
print("package:", PACKAGE)
print("pandas:", pd.__version__)

root: /home/drwho1985/workspace/soccer/beijing-lot/.worktrees/t_c83ca9dd
package: /home/drwho1985/workspace/soccer/beijing-lot/.worktrees/t_c83ca9dd/results/bj232_divergence_support_v1
pandas: 3.0.3


## Build the package

`builder.build` assembles the full evidence set: source census, loss-of-
support waterfall (REQ-S06), zero-overlap diagnosis, decision-epoch SP
triplet recovery separate from de-vig (REQ-S05), independent-source
reconciliation, issue-level modelability (REQ-S08) and #229 debt E1-E4
closure (REQ-S09). The package is published atomically.

The E1 `AUDITED_SHA` is a placeholder here; EV review supplies the exact
reviewed HEAD before finalize.

In [2]:
receipt = builder.build(ROOT)
target = builder.write_package(receipt, audited_sha="0" * 40, force=True)
print("BUILT", target)
print("disposition:", receipt.disposition)
print("evidence rows:", len(receipt.evidence))
print("debt closure all_closed:", receipt.debt_closure["all_closed"])

/home/drwho1985/workspace/soccer/beijing-lot/.worktrees/t_c83ca9dd/src/divergence_support/waterfall.py:485: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  power = power.merge(
/home/drwho1985/workspace/soccer/beijing-lot/.worktrees/t_c83ca9dd/src/divergence_support/waterfall.py:485: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  power = power.merge(


/home/drwho1985/workspace/soccer/beijing-lot/.worktrees/t_c83ca9dd/src/divergence_support/reconcile.py:101: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  power["power_cutoff_ts_utc"] = power_ts


BUILT /home/drwho1985/workspace/soccer/beijing-lot/.worktrees/t_c83ca9dd/results/bj232_divergence_support_v1
disposition: UNRECOVERABLE_WITH_CURRENT_SOURCES
evidence rows: 10
debt closure all_closed: True


/home/drwho1985/workspace/soccer/beijing-lot/.venv/lib/python3.14/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


## Source census

Committed upstream packages are SHA256-verified. Raw recoverable assets
(official 3SP export, live collector snapshots, sp-drift panel, 500.bjdc
live captures) are gitignored on the producer machine: they are reported
present with hashes when available and ABSENT otherwise. No conclusion
depends on a raw asset being present.

In [3]:
display(receipt.census[["key", "present", "rows", "issue_min", "issue_max", "clock"]])
print("raw assets absent in this checkout:", receipt.census.loc[~receipt.census["present"], "key"].tolist())

,key,present,rows,issue_min,issue_max,clock
0,bj214_selected_market_observations,True,16496,26051,26083,snapshot_ts_utc
1,full_3sp_official,False,0,NaN,NaN,last_update_ts
2,beidan_live_snapshots,False,0,NaN,NaN,snapshot_ts
3,sp_drift_panel,False,0,NaN,NaN,snapshot_ts
4,500_bjdc_live_sp_snapshots,False,0,NaN,NaN,fetched_at
5,bj203_devig_euro_probability,True,332718,190101,26069,euro_1h_available_ts_utc
6,bj204_avg_eu,True,357705,190101,26049,euro_1h_available_ts_utc
7,bj180_match_probability_freeze,True,1558,20012,25042,feature_cutoff_ts


raw assets absent in this checkout: ['full_3sp_official', 'beidan_live_snapshots', 'sp_drift_panel', '500_bjdc_live_sp_snapshots']


## Loss-of-support waterfall (REQ-S06)

Every source family is traced RAW -> IDENTITY_JOIN -> TIMESTAMP_GATE ->
COMPLETE_TRIPLET -> CAUSAL_INTERSECTION.  Rows are counted through
COMPLETE_TRIPLET; the CAUSAL_INTERSECTION stage expands to canonical
3/1/0 cells (EXPAND row) and counts cell losses.  The waterfall is a
replayable ledger: RAW - sum(DROPs) == rows, rows*factor == EXPAND,
EXPAND - sum(cell DROPs) == final cells.  The verifier replays it.

In [4]:
display(receipt.waterfall)
display(receipt.waterfall_reasons)

,source,source_key,stage,count,kind,unit,reason
0,Q_BJ_SP_DEVIG,bj214_selected_market_observations,RAW_ROWS,16496,PASS,row,
1,Q_BJ_SP_DEVIG,bj214_selected_market_observations,IDENTITY_JOIN,5332,DROP,row,NO_IDENTITY_MATCH
2,Q_BJ_SP_DEVIG,bj214_selected_market_observations,IDENTITY_JOIN,11164,PASS,row,
3,Q_BJ_SP_DEVIG,bj214_selected_market_observations,TIMESTAMP_GATE,7664,DROP,row,POST_DECISION_OBSERVATION
4,Q_BJ_SP_DEVIG,bj214_selected_market_observations,TIMESTAMP_GATE,3500,PASS,row,
5,Q_BJ_SP_DEVIG,bj214_selected_market_observations,COMPLETE_TRIPLET,1174,DROP,row,PARTIAL_SOURCE_TRIPLET
6,Q_BJ_SP_DEVIG,bj214_selected_market_observations,COMPLETE_TRIPLET,154,DROP,row,DUPLICATE_SOURCE_ROW
7,Q_BJ_SP_DEVIG,bj214_selected_market_observations,COMPLETE_TRIPLET,2172,PASS,row,
8,Q_BJ_SP_DEVIG,bj214_selected_market_observations,CAUSAL_INTERSECTION,6516,EXPAND,cell,AXIS_EXPAND_3_OUTCOMES
9,Q_BJ_SP_DEVIG,bj214_selected_market_observations,CAUSAL_INTERSECTION,6516,DROP,cell,NOT_IN_MODELING_AXIS


,reason,count
5,POST_DECISION_OBSERVATION,334782
2,NOT_IN_MODELING_AXIS,6516
3,NO_IDENTITY_MATCH,5332
4,PARTIAL_SOURCE_TRIPLET,5278
0,ANCHOR_UNAVAILABLE,3210
1,DUPLICATE_SOURCE_ROW,154
6,RAW_ASSET_ABSENT,0


## Zero-overlap diagnosis

Classifies WHY each modelability interface has zero causal support, with
the evidence columns behind the classification.

In [5]:
display(receipt.diagnosis[["interface", "root_cause", "evidence"]])

,interface,root_cause,evidence
0,Q_BJ_x_SP,GENUINELY_UNAVAILABLE_HISTORICAL_DATA,#214 SP observations cover issues 26051..26083...
1,Q_BJ_x_EU_x_SP,GENUINELY_UNAVAILABLE_HISTORICAL_DATA,EU: EU de-vig causal support exists on histori...
2,Q_BJ_x_POWER_x_SP,GENUINELY_UNAVAILABLE_HISTORICAL_DATA,POWER: P_POWER_BJLINE causal cells exist on hi...


## Decision-epoch SP triplets separate from de-vig (REQ-S05)

Recovered triplets carry raw SP prices (SP_3/1/0) and the normalized
inverse (Q_BJ_SP_DEVIG) as distinct columns, with observed-at, decision
epoch and the proof that de-vig == normalize(1/SP) on every row.  When
nothing is causally recoverable, an explicit empty receipt is published.

In [6]:
display(receipt.recovery_receipts)
if len(receipt.recovery_sp_triplets):
    display(receipt.recovery_sp_triplets.head(10))
    display(receipt.recovery_relationship)

,interface,recovered_triplet_cells,recovered_issue_count,reason
0,Q_BJ_SP_DEVIG,2223,17,RECOVERED


,issue,match_idx,year,SP_3,SP_1,SP_0,Q_BJ_SP_DEVIG_3,Q_BJ_SP_DEVIG_1,Q_BJ_SP_DEVIG_0,observed_at_utc,decision_epoch_utc,kickoff_ts_utc,source_identity,causal,devig_matches_normalized_inverse,triplet_sum,triplet_bounded
0,26051,294,2026,2.77,4.33,2.44,0.360364,0.230533,0.409102,2026-05-04 01:37:59+00:00,2026-05-04 01:38:05+00:00,2026-05-03T14:15:00Z,bj214_decision_epoch_observation,True,True,1.0,True
1,26051,293,2026,1.96,2.99,6.31,0.508612,0.333404,0.157984,2026-05-04 01:37:59+00:00,2026-05-04 01:38:05+00:00,2026-05-03T14:15:00Z,bj214_decision_epoch_observation,True,True,1.0,True
2,26051,292,2026,2.46,4.22,2.80,0.406255,0.236821,0.356924,2026-05-04 01:37:59+00:00,2026-05-04 01:38:05+00:00,2026-05-03T14:00:00Z,bj214_decision_epoch_observation,True,True,1.0,True
3,26051,291,2026,2.96,4.22,2.34,0.337111,0.236457,0.426431,2026-05-04 01:37:59+00:00,2026-05-04 01:38:05+00:00,2026-05-03T14:00:00Z,bj214_decision_epoch_observation,True,True,1.0,True
4,26051,290,2026,2.91,3.70,2.58,0.343125,0.269863,0.387013,2026-05-04 01:37:59+00:00,2026-05-04 01:38:05+00:00,2026-05-03T14:00:00Z,bj214_decision_epoch_observation,True,True,1.0,True
5,26051,289,2026,2.88,3.57,2.67,0.346575,0.279590,0.373834,2026-05-04 01:37:59+00:00,2026-05-04 01:38:05+00:00,2026-05-03T14:00:00Z,bj214_decision_epoch_observation,True,True,1.0,True
6,26051,288,2026,1.89,3.79,4.81,0.528650,0.263627,0.207723,2026-05-04 01:37:59+00:00,2026-05-04 01:38:05+00:00,2026-05-03T14:00:00Z,bj214_decision_epoch_observation,True,True,1.0,True
7,26051,287,2026,2.85,4.10,2.46,0.350427,0.243590,0.405983,2026-05-04 01:37:59+00:00,2026-05-04 01:38:05+00:00,2026-05-03T13:30:00Z,bj214_decision_epoch_observation,True,True,1.0,True
8,26051,286,2026,3.07,3.58,2.52,0.325119,0.278803,0.396078,2026-05-04 01:37:59+00:00,2026-05-04 01:38:05+00:00,2026-05-03T13:30:00Z,bj214_decision_epoch_observation,True,True,1.0,True
9,26051,285,2026,4.37,5.52,1.69,0.228443,0.180850,0.590707,2026-05-04 01:37:59+00:00,2026-05-04 01:38:05+00:00,2026-05-03T13:00:00Z,bj214_decision_epoch_observation,True,True,1.0,True


,issue,year,triplet_count,devig_matches,all_bounded
0,26051,2026,108,True,True
1,26052,2026,1,True,True
2,26053,2026,32,True,True
3,26054,2026,200,True,True
4,26055,2026,33,True,True
5,26056,2026,58,True,True
6,26057,2026,40,True,True
7,26058,2026,11,True,True
8,26061,2026,26,True,True
9,26062,2026,12,True,True


## Independent-source reconciliation

POWER and EU are reconciled against the Q_BJ anchor on the identical
issue/match/native-line/outcome space.  Only causally available cells on
both sides are published; same-source alias rows are never treated as
independent.

In [7]:
display(receipt.reconcile_receipts)

,interface,reconciled_cells,reconciled_issues,line_reconciled,translation_available
0,Q_BJ_x_POWER_BJLINE,0,0,4674.0,NaN
1,Q_BJ_x_EU_BJLINE,0,0,NaN,192795.0


## Issue-level modelability (REQ-S08)

Complete matched issues per chronological cohort (fit <= 2021, tune 2022,
calibration 2023) and feasible-N support on the primary Q_BJ x SP axis.

In [8]:
display(receipt.modelability["interface_summary"])
display(receipt.modelability["n_support_qbj_sp"])
print("issue registry rows:", len(receipt.modelability["issue_registry"]))

,interface,fit_issues,fit_matches,tune_issues,tune_matches,calibration_issues,calibration_matches,2024_plus_issues,2024_plus_matches
0,Q_BJ_x_SP,0,0,0,0,0,0,0,0
1,Q_BJ_x_EU_x_SP,0,0,0,0,0,0,0,0
2,Q_BJ_x_POWER_x_SP,0,0,0,0,0,0,0,0


,n,issues_with_ge_n,fraction_of_issues
0,1,0,0.0


issue registry rows: 394


## REQ-S evidence and #229 debt closure

In [9]:
display(receipt.evidence[["requirement", "name", "pass"]])
print("debt closure:", {k: v for k, v in receipt.debt_closure.items() if k != "e1_binding"})

,requirement,name,pass
0,REQ-S01,Source identity and causal observed-at proof,True
1,REQ-S02,Identity reconciliation without silent drops,True
2,REQ-S03,Probability triplet validity,True
3,REQ-S04,Common native handicap outcome space,True
4,REQ-S05,SP separate from probability,True
5,REQ-S06,Support waterfall reconstruction,True
6,REQ-S07,Label boundary sealed,True
7,REQ-S08,Issue-level modelability,True
8,REQ-S09,#229 debt E1-E4 closure,True
9,REQ-S10,Reviewed-HEAD binding,True


debt closure: {'e2_schema_problems': [], 'e3_journal_consistent': True, 'e3_crash_restart_probe': True, 'e4_protocol_problems': [], 'all_closed': True}


## Verify the immutable package

Hash verification plus independent semantic checks (evidence, schema,
debt E1-E4, waterfall replay, binding).

In [10]:
import subprocess
result = subprocess.run(
    [sys.executable, "scripts/build_bj232_divergence_support.py", "verify",
     "--package", "bj232_divergence_support_v1"],
    cwd=ROOT, capture_output=True, text=True,
)
print(result.stdout)
if result.returncode:
    print(result.stderr)
print("verify exit:", result.returncode)

VERIFIED /home/drwho1985/workspace/soccer/beijing-lot/.worktrees/t_c83ca9dd/results/bj232_divergence_support_v1: 27 files match SHA256SUMS.json
SEMANTIC CHECKS PASS (evidence/schema/debt/waterfall/binding)

verify exit: 0


## Terminal disposition

The modelability gate requires positive causal `Q_BJ x SP` support AND
positive `Q_BJ x EU x SP` or `Q_BJ x POWER x SP` support on complete
triplets across fit/tune/calibration. The published evidence determines
the disposition; no automatic threshold beyond support > 0 is used.

In [11]:
print("TERMINAL DISPOSITION:", receipt.disposition)
print("recovered SP issues:", receipt.recovery_receipts["recovered_issue_count"].sum() if len(receipt.recovery_receipts) else 0)
print("Q_BJ x SP fit/tune/cal issues:", int(receipt.modelability["interface_summary"][receipt.modelability["interface_summary"]["interface"].eq("Q_BJ_x_SP")][["fit_issues", "tune_issues", "calibration_issues"]].sum().sum()))

TERMINAL DISPOSITION: UNRECOVERABLE_WITH_CURRENT_SOURCES
recovered SP issues: 17
Q_BJ x SP fit/tune/cal issues: 0
